**Does it all: Phase1a numbers and 1b numbers and cluster labelled circles (for DroneShort1 Half Resolution)**

Unfortunately, it takes about 1 sec/frame, and over (it seems) 6GB memory.  So I started search of wand documentation to try to get it to do the current numpy calcs..save one suitable exclusion zone image, wand combine
it with original, and then wand draw numbers, arrows and circle(s) as already done.

Idea: Use Imagemagick to create images that JUST contain the annotations (text and graphic), so each will be very small.

Then, we can use gimp to layer the two relevent thumb..bmp images and the annotation images. 

Then we can use gimp to shift from image to image, and calculate the difference and subtraction images.  

In [56]:
import numpy as np

In [57]:
import wand
from wand.image import Image
from wand.drawing import Drawing

In [58]:
import math

In [59]:
import matplotlib.pyplot as plt

In [60]:
from os import getcwd

**Numerical Data**

*Phase1a .int file output*

*.int line of 26 numbers*

In [ ]:
dtint = np.dtype([('frame', np.int32), 
               ('Rd','<i8'),('Rx',np.int16),('Ry',np.int16),
               ('Gd','<i8'),('Gx',np.int16),('Gy',np.int16),
               ('Bd','<i8'),('Bx',np.int16),('By',np.int16),
               ('rd','<i8'),('rx',np.int16),('ry',np.int16),
               ('gd','<i8'),('gx',np.int16),('gy',np.int16),
               ('bd','<i8'),('bx',np.int16),('by',np.int16),
               ('Rn',np.int32),('Gn',np.int32),('Bn',np.int32),
               ('rn',np.int32),('gn',np.int32),('bn',np.int32),
               ('stn',np.int32)]
             )

*Phase 1b .out file output*
  
.out line of 6 numbers*

In [ ]:
dtout = np.dtype([('cluster', np.int32), ('frame', np.int32),
                ('diff','<i8'),
                ('x',np.int16),('y',np.int16),
                ('score',np.float16)])

In [ ]:
BMD='/media/seth/CTAP/bmdir/bitmaps-jobHF'
RED='/data/GIT/C-TAP/RESULTS-jobHF'
OutPath=getcwd()+"/VisPhase1"
MOVN='DS'
RUNUM='1'
FirstFrameN=1
UNMASKFILE=RED+"/"+MOVN+".btmask."+RUNUM
INTFILE=   RED+"/"+MOVN+".int."+RUNUM
OUTFILE=   RED+"/"+MOVN+".out."+RUNUM
print(BMD,RED,OutPath,UNMASKFILE,INTFILE,OUTFILE,sep='\n')

In [ ]:
intdata = np.loadtxt(INTFILE,converters=float,dtype=dtint)
nframes=len(intdata)
outdata = np.loadtxt(OUTFILE,converters=float,dtype=dtout)

**Functions**

In [ ]:
def getintrow(fn):
    row=np.array(intdata[fn-1])
    #print("row=",row)
    #print()
    t=intdata[fn-1].item(0)
    intdatarowstr = str(t[0])
    for z in range(1,24,3):
        intdatarowstr+="  "+str(t[z:z+3])
    intdatarowstr+="   "+str(t[24])
    return {"string": intdatarowstr, "row": row}

In [ ]:
def getanyoutrow(frameno):
    # if there is a row in .out data with given frame number,
    # return list [  string of text to draw on image,
    #                the row (ndarray structure) for drawing circle on image ]
    # otherwise, None is returned
    row=outdata[  outdata[:]['frame']==frameno   ]
    #print("row=",row)
    if len(row) > 0:
        if len(row) > 1 :
            print("Somethings wrong. ", len(row), " have frame number ", frameno)
            print(row)
        r = row[0]
        #print(r)
        #print(type(r))
        ret={'string': str(r['cluster'])  + "     " 
             + str(r['frame']) + "      ("
             + str(r['x'] ) + ", " 
             + str(r['y']) + ")     " 
             + str(r['score']),
              'row': row[0]}
    else:
            ret=None
    #print(ret)
    #if ret != None :
    #    print(ret[0])
    #    print(type(ret[0]))
    return ret

In [ ]:
getanyoutrow(14)

In [ ]:
def to6( n ):
    return "{!s:>06}".format(n)
def tothumb(n):
    return BMD+"/thumb"+to6(n)+".bmp"
#tothumb(1234)

def getwh(n):
    img=Image(filename=tothumb(1))   
    return img.width,img.height

width,height=getwh(1)
widthTo1920=math.ceil(float(width)/1920.0)

In [ ]:
#This makes an ordinary ndarray that will be used as the mask when we make
# np.ma masked arrays for doing the statistical calculations.

def makemaskarray(f): 
    unmaskbits=np.fromfile(f,dtype=np.uint8) 
        #the bits left-to-right in the bytes of the file and array correspond to pixels we that will not be masked.
    unmaskbytes=np.unpackbits(unmaskbits.data)
        #convert to an array of bytes valued 0 or 1
    unmaskbytesrect=np.reshape(unmaskbytes,(height,width)) #make into rectangle
    unmaskRGBrect0=np.repeat(unmaskbytesrect,3,axis=1)     #triple each 1 or 0 for masking R B B
    unmaskRGBrect1=np.reshape(unmaskRGBrect0,(height,width,3)) 
    #group each threesome for a color the plot software will like
    maskRGBrect=1-unmaskRGBrect1 #convert 1s for using to 0 for numpy's not masking
    return maskRGBrect

maskRGBrect=makemaskarray(UNMASKFILE)
#maskRGBrect

In [ ]:
def getimgs(n):
    bmpfile=tothumb(n)                                         #filename
    ba=np.fromfile(bmpfile,dtype=np.uint8).astype(np.int16)    #ndarray
    #Use 16 bits instead of 8 so subtraction is ok.
    #WARNING-To export a .bmp we must convert to uint8
    bh=np.array(ba[0:54],dtype=np.uint8)                           #.bmp header
    #WARNING-To export a .bmp we must convert to uint8
    ba=np.flip(ba[54:].reshape([height,width,3]),(0,2))        #.bmp data, widthXheightX{RGB}
    #First, reshape groups into triples the bytes each row.
    #Second, flip (0) flips Microsoft orientation right side up; and (2) flips GBR to RGB
    #shape=b1.shape
    #zeros=np.zeros(shape,like=b1,dtype=np.int16) #for devel experiment
    
    #continue with mask

    mba=np.ma.masked_array(ba,mask=maskRGBrect,fill_value=128) #Use gentle gray to display masked img.
    #so they don't see the fill value 
    #mb1filled=mb1.filled()
    #mb2filled=mb2.filled()

    return (mba.filled(),ba,bh,mba)
    #  (
    #    [0] RGB array of int16 with excluded pixels colored (128,128,128),
    #    [1] original RGB array of int16,
    #    [2] .bmp header, uint8 ready to append to a uint8 BGR to export a .bmp file,
    #    [3] masked RGB of int16; it has the mask of excluded pixels and fill_value=128 
    #      (but plt.imshow will ignore the masking, we had to .filled() it for plt.imshow to show exclusions)
    #  )


In [ ]:
baez,ba,bh,mba=getimgs(120)

In [ ]:
#plt.imshow(baez)
plt.imshow(ba)
#ba
#bh
#mba

In [ ]:
def skywithez(n):
    _,ba,_,_=getimgs(n)
    fm1s=np.ones_like(ba,dtype=float)
    fmasked1=np.ma.masked_array(fm1s,mask=maskRGBrect,fill_value=[0.8,0.0,0.0]).filled()
    baf=np.array(ba,dtype=float)/255.0
    return fmasked1*baf
    #filled=np.ma.masked_array(fm1,mask=maskRGBrect,fill_value=[0.3,0.0,0.0]).filled()

In [ ]:
skywithezret=skywithez(120)
plt.imshow(skywithezret)

Setup for drawing the data lines.

In [ ]:
#text line locations
llcapx=int(width/20)
llcapy=int(height-width/20)
llcapyabove=int(height-1.5*width/20)

Setup for drawing data line

In [ ]:
dra=Drawing()        #for drawing .int data line
drcloner=dra.clone()    #for future drawings
drbC=drcloner.clone()   #for drawing .out circle

dra.font_size=30*widthTo1920
drb=dra.clone()   #for drawing .out data line, same font_size
dra.fill_color="WHITE"
dra.stroke_color="WHITE"

drb.fill_color="YELLOW"
drb.stroke_color="YELLOW"

drbC.font_size=30*widthTo1920
drbC.fill_opacity=0     #for drawing .out point circle
drbC.stroke_color="TEAL"
drbC.stroke_width=6

Setup for drawing markers on Phase1a selected pixels.

In [ ]:
def M(TH) :
    return( np.array( [ [math.cos(math.pi*TH/180.), math.sin(math.pi*TH/180.)], [-math.sin(math.pi*TH/180.), math.cos(math.pi*TH/180.)] ] ) )

#Geometry of normalized unit arrows to show RGB changes
TH=30.0                  #angle of arrows away from vertical, and hands away from body
lcircr=0.1               #little circle radius
hslen=0.1                #length of each hand of an
# Unit Vectors
g1=np.array([0.0,1.0])   #unit lower case, down, green

#tiny vectors
tc=lcircr*np.array([0.0,1.0]) #radius (down, y dir of tiny circle, and foot of down unit arrow
def T(s) :
    return (tc + g1*s) #tail on tiny circle, head down by unit * s (scale, 0<=s<=1)

#Arrow body is (tc->T(s)*M..
def arrB(s) :
    return np.array([tc, T(s)])

TL=hslen*g1@M(180.0+TH) #coord of left hand rel to head
TR=hslen*g1@M(180.0-TH) #coord of right hand rel to head

#down dir Left arm LA(s) is (T(s)->TL(s))
def arrL(s) :
    return np.array([T(s), T(s)+TL])


#down dir Right arm LA(s) is (T(s)->TL(s))
def arrR(s) :
    return np.array([T(s), T(s)+TR])

def unit_up_arrow(s) :
    return -np.concat([arrB(s),arrL(s),arrR(s)])


In [ ]:
def dispdata(row):
    #print(fn, row['frame'])
    return ( [ row['Rd'],   -30.0, [row['Rx'],row['Ry']], row['Rn' ], "red" ],
             [ row['Gd'],     0.0, [row['Gx'],row['Gy']], row['Gn' ], "green" ],
             [ row['Bd'],    30.0, [row['Bx'],row['By']], row['Bn' ], "blue" ],
             [ -row['rd'], -150.0, [row['rx'],row['ry']], row['rn' ], "red" ],
             [ -row['gd'],  180.0, [row['gx'],row['gy']], row['gn' ], "green" ],
             [ -row['bd'],  150.0, [row['rx'],row['by']], row['bn' ], "blue" ] )             

In [ ]:
colvaldiv = float(128)
arrlen = 100*widthTo1920

def drdata(dwg, row):
    data = dispdata(row)
    for r in data:
        #print(r)
        #print((r[0]/colvaldiv))
        #print( "arrow", arrlen*unit_up_arrow( (r[0]/colvaldiv)) )
        line = np.round( (arrlen*unit_up_arrow(r[0]/colvaldiv))@M(r[1]) + r[2]).astype(int)
        dwg.stroke_width = 2
        dwg.stroke_color = wand.color.Color( r[4] )
        for i in range(0,3):
            dwg.line(line[2*i],line[2*i+1])

In [ ]:
#img=0
def vis(fn):
    global img
    imga=skywithez(fn)
    imga=np.flip(imga,(0,2)) #RGB-->BGR 
    imga=np.array(imga*256,dtype=np.uint8).flatten()
    imga=np.concat((bh,imga))
    img=Image(blob=imga)  
    row=np.array(intdata[fn-1])
    t=intdata[fn-1].item(0)
    intdatarowstr = str(t[0])
    for z in range(1,24,3):
        intdatarowstr+=" "+str(t[z:z+3])
    intdatarowstr+="  "+str(t[24])
    
    draw=dra.clone()
    draw.text(llcapx,llcapy,intdatarowstr)
    drdata(draw, row )
    draw(img)
    outrow=getanyoutrow(fn)
    if outrow :
        text=outrow['string']
        draw=drb.clone()
        draw.text(llcapx,llcapyabove,text)
        draw(img)
        data=outrow['row']
        draw=drbC.clone()
        draw.circle((data[3],data[4]),(data[3]+30,data[4]))
        draw.text(data[3]+40,data[4],str(data[0]))
        draw(img)
    return img #so jupyter tries to print the result which makes the picture appear!      

In [ ]:
imgret=vis(120)

In [ ]:
imgret

In [ ]:
img.flip()

In [ ]:
img.composite?

In [ ]:
wand.image.COMPOSITE_OPERATORS

In [ ]:
def framefilename(n):
    return OutPath+"/VisPhase1"+"/frame"+to6(n)+".jpg"
def doAll():
    for i in range(nframes):
        fn=i+1
        img=vis(fn)
        print("visualized frame", fn, end="")
        img.format = 'jpeg'
        img.save(filefilename=(fn))
        print(" saved", fn, end="\r")

In [ ]:
doAll()

